# Assignment 2

In this assigment, we will work with the *Forest Fire* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/162/forest+fires). Extract the data files into the subdirectory: `../data/fires/` (relative to `./05_src/`).

## Objective

+ The model objective is to predict the area affected by forest fires given the features set. 
+ The objective of this exercise is to assess your ability to construct and evaluate model pipelines.
+ Please note: the instructions are not meant to be 100% prescriptive, but instead they are a set of minimum requirements. If you find predictive performance gains by applying additional steps, by all means show them. 

## Variable Description

From the description file contained in the archive (`forestfires.names`), we obtain the following variable descriptions:

1. X - x-axis spatial coordinate within the Montesinho park map: 1 to 9
2. Y - y-axis spatial coordinate within the Montesinho park map: 2 to 9
3. month - month of the year: "jan" to "dec" 
4. day - day of the week: "mon" to "sun"
5. FFMC - FFMC index from the FWI system: 18.7 to 96.20
6. DMC - DMC index from the FWI system: 1.1 to 291.3 
7. DC - DC index from the FWI system: 7.9 to 860.6 
8. ISI - ISI index from the FWI system: 0.0 to 56.10
9. temp - temperature in Celsius degrees: 2.2 to 33.30
10. RH - relative humidity in %: 15.0 to 100
11. wind - wind speed in km/h: 0.40 to 9.40 
12. rain - outside rain in mm/m2 : 0.0 to 6.4 
13. area - the burned area of the forest (in ha): 0.00 to 1090.84 









### Specific Tasks

+ Construct four model pipelines, out of combinations of the following components:

    + Preprocessors:

        - A simple processor that only scales numeric variables and recodes categorical variables.
        - A transformation preprocessor that scales numeric variables and applies a non-linear transformation.
    
    + Regressor:

        - A baseline regressor, which could be a [K-nearest neighbours model]() or a linear model like [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) or [Ridge Regressors](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ridge_regression.html).
        - An advanced regressor of your choice (e.g., Bagging, Boosting, SVR, etc.). TIP: select a tree-based method such that it does not take too long to run SHAP further below. 

+ Evaluate tune and evaluate each of the four model pipelines. 

    - Select a [performance metric](https://scikit-learn.org/stable/modules/linear_model.html) out of the following options: explained variance, max error, root mean squared error (RMSE), mean absolute error (MAE), r-squared.
    - *TIPS*: 
    
        * Out of the suggested metrics above, [some are correlation metrics, but this is a prediction problem](https://www.tmwr.org/performance#performance). Choose wisely (and don't choose the incorrect options.) 

+ Select the best-performing model and explain its predictions.

    - Provide local explanations.
    - Obtain global explanations and recommend a variable selection strategy.

+ Export your model as a pickle file.


You can work on the Jupyter notebook, as this experiment is fairly short (no need to use sacred). 

# Load the data

Place the files in the ../../05_src/data/fires/ directory and load the appropriate file. 

In [13]:
# Load the libraries as required.
import os
os.getcwd()
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

In [3]:
# Load data
columns = [
    'coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain', 'area' 
]
fires_dt = (pd.read_csv('../../05_src/data/fires/forestfires.csv', header = 0, names = columns))
fires_dt.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   coord_x  517 non-null    int64  
 1   coord_y  517 non-null    int64  
 2   month    517 non-null    object 
 3   day      517 non-null    object 
 4   ffmc     517 non-null    float64
 5   dmc      517 non-null    float64
 6   dc       517 non-null    float64
 7   isi      517 non-null    float64
 8   temp     517 non-null    float64
 9   rh       517 non-null    int64  
 10  wind     517 non-null    float64
 11  rain     517 non-null    float64
 12  area     517 non-null    float64
dtypes: float64(8), int64(3), object(2)
memory usage: 52.6+ KB


# Get X and Y

Create the features data frame and target data.

In [4]:
fires_dt #check data

,coord_x,coord_y,month,day,ffmc,dmc,dc,isi,temp,rh,wind,rain,area
0,7,5,mar,fri,86.2,26.2,94.3,5.1,8.2,51,6.7,0.0,0.00
1,7,4,oct,tue,90.6,35.4,669.1,6.7,18.0,33,0.9,0.0,0.00
2,7,4,oct,sat,90.6,43.7,686.9,6.7,14.6,33,1.3,0.0,0.00
3,8,6,mar,fri,91.7,33.3,77.5,9.0,8.3,97,4.0,0.2,0.00
4,8,6,mar,sun,89.3,51.3,102.2,9.6,11.4,99,1.8,0.0,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
512,4,3,aug,sun,81.6,56.7,665.6,1.9,27.8,32,2.7,0.0,6.44
513,2,4,aug,sun,81.6,56.7,665.6,1.9,21.9,71,5.8,0.0,54.29
514,7,4,aug,sun,81.6,56.7,665.6,1.9,21.2,70,6.7,0.0,11.16
515,1,4,aug,sat,94.4,146.0,614.7,11.3,25.6,42,4.0,0.0,0.00


In [5]:
X = fires_dt.iloc[:,:-1]
y = fires_dt['area']

# Preprocessing

Create two [Column Transformers](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), called preproc1 and preproc2, with the following guidelines:

- Numerical variables

    * (Preproc 1 and 2) Scaling: use a scaling method of your choice (Standard, Robust, Min-Max). 
    * Preproc 2 only: 
        
        + Choose a transformation for any of your input variables (or several of them). Evaluate if this transformation is convenient.
        + The choice of scaler is up to you.

- Categorical variables: 
    
    * (Preproc 1 and 2) Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) where appropriate.


+ The only difference between preproc1 and preproc2 is the non-linear transformation of the numerical variables.
    


### Preproc 1

Create preproc1 below.

+ Numeric: scaled variables, no other transforms.
+ Categorical: one-hot encoding.

In [46]:
from sklearn.compose import ColumnTransformer
preproc1 = ColumnTransformer(
    transformers=[
        ('numeric_transformer', StandardScaler(), ['coord_x','coord_y','ffmc','dmc','dc','isi','temp','rh','wind','rain']),
        ('onehot', OneHotEncoder(handle_unknown='ignore'), ['month', 'day']), 
    ]
)

### Preproc 2

Create preproc1 below.

+ Numeric: scaled variables, non-linear transformation to one or more variables.
+ Categorical: one-hot encoding.

In [124]:
from sklearn.preprocessing import FunctionTransformer
preproc2 = ColumnTransformer(
    transformers=[
        ('numeric_transformer', StandardScaler(), ['coord_x','coord_y','ffmc','dmc','dc','isi','rh','wind','rain']),
        ('log', FunctionTransformer(np.log1p), ['rain']), #log transform  on temperature to see if it makes a difference
        ('onehot', OneHotEncoder(handle_unknown='ignore'), ['month', 'day']), 
    ]
)

## Model Pipeline


Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `regressor` and assign a regression model to it. 

## Regressor

+ Use a regression model to perform a prediction. 

    - Choose a baseline regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Choose a more advance regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Both model choices are up to you, feel free to experiment.

In [128]:
# Pipeline A = preproc1 + baseline
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
pipeA = Pipeline(
    [
        ('preprocessing', preproc1), 
        ('regressor', KNeighborsRegressor(n_neighbors=5))
    ]
)

In [125]:
# Pipeline B = preproc2 + baseline
pipeB = Pipeline(
    [
    ('preprocessing', preproc2), 
    ('regressor', KNeighborsRegressor(n_neighbors=5))
    ]
)

In [127]:
# Pipeline C = preproc1 + advanced model
pipeC = Pipeline(
    [
        ('preprocessing', preproc1), 
        ('regressor', DecisionTreeRegressor(max_depth=5))
    ]
)

In [126]:
# Pipeline D = preproc2 + advanced model
pipeD = Pipeline(
    [
        ('preprocessing', preproc2), 
        ('regressor', DecisionTreeRegressor(max_depth=5))
    ]
)

# Tune Hyperparams

+ Perform GridSearch on each of the four pipelines. 
+ Tune at least one hyperparameter per pipeline.
+ Experiment with at least four value combinations per pipeline.

In [140]:
from sklearn.model_selection import GridSearchCV
param_grid_A = {'regressor__n_neighbors': [3, 5, 10, 15]}
scoring = ['neg_mean_squared_error', 'r2', 'neg_root_mean_squared_error']
grid_cv_A = GridSearchCV(
    estimator=pipeA, 
    param_grid=param_grid_A, 
    scoring = scoring, 
    cv = 5,
    refit = "neg_mean_squared_error")
grid_cv_A.fit(X, y)
best_params_A = grid_cv_A.best_params_
best_score_A = grid_cv_A.best_score_

In [141]:
param_grid_B = {'regressor__n_neighbors': [3, 5, 10, 15]}
grid_cv_B = GridSearchCV(
    estimator=pipeB, 
    param_grid=param_grid_B, 
    scoring = scoring, 
    cv = 5,
    refit = "neg_mean_squared_error")
grid_cv_B.fit(X, y)
best_params_B = grid_cv_B.best_params_
best_score_B = grid_cv_B.best_score_

In [145]:
param_grid_C = {'regressor__max_depth': [5, 10, 15, 20]}
grid_cv_C = GridSearchCV(
    estimator=pipeC, 
    param_grid=param_grid_C, 
    scoring = scoring, 
    cv = 5,
    refit = "neg_mean_squared_error")
grid_cv_C.fit(X, y)
best_params_C = grid_cv_C.best_params_
best_score_C = grid_cv_C.best_score_

In [146]:
param_grid_D = {'regressor__max_depth': [5, 10, 15, 20]}
grid_cv_D = GridSearchCV(
    estimator=pipeD, 
    param_grid=param_grid_D, 
    scoring = scoring, 
    cv = 5,
    refit = "neg_mean_squared_error")
grid_cv_D.fit(X, y)
best_params_D = grid_cv_D.best_params_
best_score_D = grid_cv_D.best_score_

# Evaluate

+ Which model has the best performance?

In [148]:
print(f'A: {best_score_A} \nB: {best_score_B} \nC: {best_score_C} \nD: {best_score_D}')
#model A looks to be the best
#print model A's estimator
print(best_params_A)

A: -4666.792734774268 
B: -4676.821389174342 
C: -4887.325104813472 
D: -14403.548222759779
{'regressor__n_neighbors': 15}


# Export

+ Save the best performing model to a pickle file.

In [149]:
import pickle

best_model = grid_cv_A.best_estimator_
with open('best_model.pkl', 'wb') as file:
    pickle.dump(best_model, file)

# Explain

+ Use SHAP values to explain the following only for the best-performing model:

    - Select an observation in your test set and explain which are the most important features that explain that observation's specific prediction.

    - In general, across the complete training set, which features are the most and least important.

+ If you were to remove features from the model, which ones would you remove? Why? How would you test that these features are actually enhancing model performance?

In [150]:
import shap
sampled_data = shap.sample(X, 50) #ran this for 50 randomly selected observations
data_transform = best_model.named_steps['preprocessing'].transform(sampled_data)

explainer = shap.KernelExplainer(
    best_model.named_steps['regressor'].predict, 
    data_transform
)

shap_values = explainer.shap_values(data_transform)

100%|██████████| 50/50 [00:35<00:00,  1.41it/s]


In [154]:
sample_idx = 0  # first observation
shap_values_sample = shap_values[sample_idx]

feature_names = best_model.named_steps['preprocessing'].get_feature_names_out()
sorted_feature_indices = np.argsort(np.abs(shap_values_sample))
sorted_feature_importance = [(feature_names[i], shap_values_sample[i]) for i in sorted_feature_indices]

for feature, importance in sorted_feature_importance:
    print(f"{feature}: {importance:.4f}")


onehot__month_jan: 0.0000
onehot__day_thu: 0.0000
onehot__day_sun: 0.0000
onehot__day_mon: 0.0000
onehot__month_nov: 0.0000
onehot__month_may: 0.0000
numeric_transformer__wind: 0.0000
onehot__month_apr: 0.0000
numeric_transformer__rain: 0.1039
onehot__month_oct: -0.1519
numeric_transformer__ffmc: -0.1851
onehot__day_wed: 0.2971
onehot__month_jul: -0.3170
onehot__month_jun: -0.3388
onehot__day_tue: 0.3411
onehot__month_dec: 0.3756
onehot__month_feb: -0.4130
onehot__day_fri: 0.4349
onehot__month_aug: 0.6588
onehot__day_sat: -0.7180
numeric_transformer__isi: 1.0192
onehot__month_sep: -1.0246
onehot__month_mar: -1.0253
numeric_transformer__rh: -1.3337
numeric_transformer__dc: -1.5622
numeric_transformer__coord_x: 1.8536
numeric_transformer__dmc: -2.2880
numeric_transformer__coord_y: -2.9870
numeric_transformer__temp: -10.3628


In [155]:
mean_abs_shap_values = np.mean(np.abs(shap_values), axis=0) #all observations
sorted_feature_indices = np.argsort(mean_abs_shap_values)
sorted_feature_importance = [(feature_names[i], mean_abs_shap_values[i]) for i in sorted_feature_indices]

for feature, importance in sorted_feature_importance:
    print(f"{feature}: {importance:.4f}")

onehot__month_jan: 0.0000
onehot__month_nov: 0.0000
onehot__month_may: 0.0000
onehot__month_apr: 0.0000
onehot__month_dec: 0.1070
onehot__month_jul: 0.1364
onehot__month_feb: 0.1388
onehot__month_jun: 0.1406
onehot__month_oct: 0.1484
onehot__month_mar: 0.1485
numeric_transformer__rain: 0.1664
onehot__day_tue: 0.1994
onehot__day_thu: 0.2459
numeric_transformer__ffmc: 0.4149
onehot__day_mon: 0.4379
onehot__day_sun: 0.6046
onehot__day_wed: 0.7645
numeric_transformer__dc: 1.0783
onehot__day_fri: 1.3270
onehot__month_aug: 1.7347
numeric_transformer__dmc: 1.9801
onehot__month_sep: 2.2182
numeric_transformer__isi: 2.2998
numeric_transformer__wind: 2.6144
onehot__day_sat: 3.0163
numeric_transformer__coord_x: 3.9201
numeric_transformer__rh: 4.7372
numeric_transformer__coord_y: 5.5307
numeric_transformer__temp: 5.9371


*(Answer here.)*

+ Selected first sample at index 0: the most important feature explaining the prediction is the temperature, followed by y-coord, dmc and x-coord

+ Across the entire set, the most important features are temperature, y and x coordinates and relative humidity rh. The least important are the jan, nov, may, apr months

+ I would remove the 4 features identified as least important (jan, nov, may, apr months) since they have shap values of 0 because they have little influence anyways. To test whether removing these affected the model, we can compute the error and compare it to what we had originally, if it stayed the same or improved, then keep them out. If not, then the feature might have been useful and we should investigate further.

## Criteria

The [rubric](./assignment_2_rubric_clean.xlsx) contains the criteria for assessment.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-2`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_2.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at the `help` channel. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Cortez,Paulo and Morais,Anbal. (2008). Forest Fires. UCI Machine Learning Repository. https://doi.org/10.24432/C5D88D.